In [1]:
# Imports
import json
from config import PROCESSED_DIR
import importlib
import lm.ngram as ngram
import lm.evaluation as evaluation
import lm.generation as generation

# Path of processed data
DATA_DIR = PROCESSED_DIR
with (DATA_DIR / "train.json").open(encoding="utf-8") as file:
    train_documents = json.load(file)
with (DATA_DIR / "validation.json").open(encoding="utf-8") as file:
    valid_documents = json.load(file)
with (DATA_DIR / "vocab.json").open(encoding="utf-8") as file:
    vocab = set(json.load(file))

In [2]:
print("Number of train books:", len(train_documents))
print("Number of validation books:", len(valid_documents))
print("Vocabulary size:", len(vocab))

print("Id of first book:", train_documents[0]["book_id"])
print("First 20 tokens:", train_documents[0]["tokens"][:20])

Number of train books: 20
Number of validation books: 1
Vocabulary size: 20000
Id of first book: 57682
First 20 tokens: ['produced', 'by', 'susan', 'skinner', ',', 'chris', '<unk>', ',', '<unk>', 'patten', 'and', 'the', 'online', 'distributed', 'proofreading', 'team', 'at', 'http', ':', '/']


In [3]:
importlib.reload(ngram)
# Bigram preparation
bigram_counts, bigram_context_counts = ngram.count_ngrams(
    train_documents, n=2
)
# Trigram preparation
trigram_counts, trigram_context_counts = ngram.count_ngrams(
    train_documents, n=3
)

print("Unique bigrams:", len(bigram_counts))
print("Unique trigrams:", len(trigram_counts))

print("Most common 10 bigrams:")
print(bigram_counts.most_common(10))

Unique bigrams: 293055
Unique trigrams: 684112
Most common 10 bigrams:
[((',', 'and'), 9937), (('of', 'the'), 8002), (('.', '"'), 5380), (('<unk>', ','), 4806), (('in', 'the'), 4305), (('-', '-'), 4053), ((',', '"'), 3671), (('to', 'the'), 3022), (('.', 'the'), 2895), ((',', 'the'), 2439)]


In [4]:
# Expected number of bigram and trigrams
expected_count = sum(
    len(document["tokens"]) + 1
    for document in train_documents
)

assert sum(bigram_counts.values()) == expected_count
assert sum(trigram_counts.values()) == expected_count

print("Total bigrams:", sum(bigram_counts.values()))
print("Total trigrams:", sum(trigram_counts.values()))

Total bigrams: 1000020
Total trigrams: 1000020


In [5]:
importlib.reload(ngram)
# Example: Chance of the word "the" appearing next to the word "of".
bigram_probability = ngram.next_word_probability(
    context=("of",),
    word="the",
    ngram_counts=bigram_counts,
    context_counts=bigram_context_counts,
)
# Example: Chance of the word "the" appearing next to the word "one" and "of".
trigram_probability = ngram.next_word_probability(
    context=("one", "of"),
    word="the",
    ngram_counts=trigram_counts,
    context_counts=trigram_context_counts,
)

print("P(the | of):", bigram_probability)
print("P(the | one, of):", trigram_probability)

P(the | of): 0.24793951787816818
P(the | one, of): 0.4859154929577465


In [6]:
importlib.reload(ngram)

print("Bigram: After 'of' ")
bigram_predictions = ngram.top_next_words(
    context=("of",),
    ngram_counts=bigram_counts,
    context_counts=bigram_context_counts,
)

for word, probability in bigram_predictions:
    print(f"{word}: %{probability * 100:.2f}")

print("\nTrigram: After 'one of' ")
trigram_predictions = ngram.top_next_words(
    context=("one", "of"),
    ngram_counts=trigram_counts,
    context_counts=trigram_context_counts,
)
for word, probability in trigram_predictions:
    print(f"{word}: %{probability * 100:.2f}")

Bigram: After 'of' 
the: %24.79
a: %4.49
his: %3.60
<unk>: %3.36
this: %1.72

Trigram: After 'one of' 
the: %48.59
them: %8.45
his: %6.87
these: %5.99
those: %4.05


In [7]:
importlib.reload(ngram)

context = ("one", "of")
word = "the"
# Real value of "the" appearing after "one" "of"
original = ngram.next_word_probability(
    context, word, trigram_counts, trigram_context_counts
)
smoothed = ngram.smoothed_probability(
    context, word, trigram_counts, trigram_context_counts, vocab
)

print("With Smoothing:", original)
print("Without Smoothing:", smoothed)

With Smoothing: 0.4859154929577465
Without Smoothing: 0.3593927004257868


In [8]:
total_probability = sum(
    ngram.smoothed_probability(
        context,
        candidate,
        trigram_counts,
        trigram_context_counts,
        vocab,
    )
    for candidate in vocab
    if candidate != "<bos>"
)

print("Sum of probabilities:", total_probability)

assert abs(total_probability - 1.0) < 1e-9

Sum of probabilities: 1.0


In [9]:
importlib.reload(evaluation)

bigram_ppl = evaluation.perplexity(
    documents=valid_documents,
    n=2,
    ngram_counts=bigram_counts,
    context_counts=bigram_context_counts,
    vocab=vocab,
    k=0.01,
)

trigram_ppl = evaluation.perplexity(
    documents=valid_documents,
    n=3,
    ngram_counts=trigram_counts,
    context_counts=trigram_context_counts,
    vocab=vocab,
    k=0.001,
)

print(f"Bigram validation perplexity: {bigram_ppl:.2f}")
print(f"Trigram validation perplexity: {trigram_ppl:.2f}")

Bigram validation perplexity: 485.25
Trigram validation perplexity: 2845.56


In [10]:
importlib.reload(generation)
# Example prompt
prompt = "it was"
# Text generation settings of bigram and trigram
bigram_text = generation.generate_text(
    prompt, 2,
    bigram_counts, bigram_context_counts, vocab,
    k=0.001,
)
trigram_text = generation.generate_text(
    prompt, 3,
    trigram_counts, trigram_context_counts, vocab,
    k=0.001,
)

print("BIGRAM:")
print(bigram_text)

print("\nTRIGRAM:")
print(trigram_text)

BIGRAM:
it was a god and the other. he was the same time to the same time the great deal of their own hands, or three or to have been able

TRIGRAM:
it was a true heart to her. he was not the same, or the other man was killed in an age when it becomes loaded with baggage and horses and
